In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import numpy as np
import os
from sklearn.model_selection import train_test_split

# ====================== 1. تحميل البيانات ======================
print("🔄 جاري تحميل البيانات...")

X = np.load('/content/drive/MyDrive/alphabet/images_under.npy')
y = np.load('/content/drive/MyDrive/alphabet/encoded_labels_under.npy')

if X.shape[-1] == 1:
    X = np.repeat(X, 3, axis=-1)

print(f"Total samples: {len(X)} | Classes: {len(np.unique(y))}")

# ====================== 2. التقسيم الصحيح ======================
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

print(f"\n✅ Test set size = {len(X_test)} samples")

# ====================== 3. حفظ الملفات في المسار الجديد ======================
save_dir = "/content/drive/MyDrive/alphabet"
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "y_test_under.npy"), y_test)
np.save(os.path.join(save_dir, "X_test_under.npy"), X_test)

print(f"✅ تم حفظ الملفات بنجاح في: {save_dir}")
print(f"   • y_test_under.npy → Shape: {y_test.shape}")
print(f"   • X_test_under.npy   → Shape: {X_test.shape}")

# ====================== تحقق ======================
print("\n📁 الملفات المحفوظة:")
print(os.listdir(save_dir))

🔄 جاري تحميل البيانات...
Total samples: 32128 | Classes: 32

✅ Test set size = 4820 samples
✅ تم حفظ الملفات بنجاح في: /content/drive/MyDrive/alphabet
   • y_test_under.npy → Shape: (4820,)
   • X_test_under.npy   → Shape: (4820, 64, 64, 3)

📁 الملفات المحفوظة:
['images_over.npy', 'images_original (1).npy', 'label_encoder_classes.npy', 'images_under.npy', 'images_original.npy', 'encoded_labels_original.npy', 'encoded_labels_over.npy', 'encoded_labels_under.npy', 'CNN_runs_analysis1.xlsx', 'mobilenet_results_under_average.xlsx', 'x_val_mobilenet_under.npy', 'CNN_Lstm_results_Over_average.xlsx', 'CNN_Lstm_results_over_average.xlsx', 'x_val_cnn_Lstm_over.npy', 'CNN_Lstm_results_original_average.xlsx', 'CNN_Lstm_analysis_Original.xlsx', 'CNN_Lstm_results_under_average.xlsx', 'mobilenet_results_original_average.xlsx', 'x_val_mobilenet_original.npy', 'x_val_mobilenet_over.npy', 'CNN_runs_analysis_Under.xlsx', 'CNN_runs_results_Under.xlsx', 'my_modelcnn_Under.keras', 'x_val_cnn_Under.npy', 'y

In [6]:
import numpy as np
import os
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, log_loss
)
from scipy.optimize import minimize
from scipy.stats import rankdata

# ====================== المسارات (مُصححة) ======================
FUSION_DIR = "/content/drive/MyDrive/fusion"
Y_DIR      = "/content/drive/MyDrive/alphabet"

PATH_CNN    = os.path.join(FUSION_DIR, "cnn_test_probs_under.npy")
PATH_MOBILE = os.path.join(FUSION_DIR, "mob_test_probs_under.npy")   # لاحظ المسافة
PATH_Y      = os.path.join(Y_DIR, "y_test_under.npy")

# ====================== تحميل البيانات ======================
p_cnn    = np.load(PATH_CNN).astype(np.float64)
p_mobile = np.load(PATH_MOBILE).astype(np.float64)
y_test   = np.load(PATH_Y).astype(np.int64)

print(f"✅ Shapes -> CNN: {p_cnn.shape} | Mobile: {p_mobile.shape} | y_test: {y_test.shape}")

n = y_test.shape[0]
p_cnn    = p_cnn[:n, :32]
p_mobile = p_mobile[:n, :32]

print(f"✅ After slicing -> CNN: {p_cnn.shape} | Mobile: {p_mobile.shape}\n")

# ====================== دالة التطبيع ======================
def normalize_probs(p):
    p = np.asarray(p, dtype=np.float64)
    row_sums = p.sum(axis=1, keepdims=True)
    return p / np.maximum(row_sums, 1e-12)

p_cnn    = normalize_probs(p_cnn)
p_mobile = normalize_probs(p_mobile)

# ====================== دالة المقاييس ======================
def compute_metrics(name, probs, y_true):
    probs = normalize_probs(probs)
    y_pred = np.argmax(probs, axis=1)

    return {
        "Method": name,
        "Val Acc": accuracy_score(y_true, y_pred),
        "Val Loss": log_loss(y_true, np.clip(probs, 1e-12, 1)),
        "Prec-W": precision_score(y_true, y_pred, average='weighted', zero_division=0),
        "Rec-W": recall_score(y_true, y_pred, average='weighted', zero_division=0),
        "F1-W": f1_score(y_true, y_pred, average='weighted', zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, probs, multi_class='ovr', average='weighted')
    }

# ====================== Ensemble Methods ======================
results = []

results.append(compute_metrics("CNN", p_cnn, y_test))
results.append(compute_metrics("MobileNetV2", p_mobile, y_test))

# Fusion Methods
soft_mean = normalize_probs((p_cnn + p_mobile) / 2)
hard_max  = normalize_probs(np.maximum(p_cnn, p_mobile))

def objective(w):
    fused = w[0]*p_cnn + w[1]*p_mobile
    return log_loss(y_test, normalize_probs(fused))

res = minimize(objective, [0.6, 0.4], bounds=[(0,1), (0,1)], method='SLSQP')
w1, w2 = res.x
print(f"🎯 Optimal Weights → CNN: {w1:.4f} | MobileNetV2: {w2:.4f}\n")

weighted   = normalize_probs(w1 * p_cnn + w2 * p_mobile)
median_f   = normalize_probs(np.median(np.stack([p_cnn, p_mobile]), axis=0))
geo_mean   = normalize_probs(np.sqrt(p_cnn * p_mobile + 1e-12))
min_rule   = normalize_probs(np.minimum(p_cnn, p_mobile))
product_f  = normalize_probs(np.prod(np.stack([p_cnn, p_mobile]), axis=0) + 1e-12)

def rank_based_fusion(probs_list):
    n_samples, n_classes = probs_list[0].shape
    rank_sum = np.zeros((n_samples, n_classes))
    for p in probs_list:
        ranks = np.apply_along_axis(lambda x: rankdata(-x, method='average'), axis=1, arr=p)
        rank_sum += ranks
    scores = np.exp(-rank_sum / 3.5)
    return normalize_probs(scores)

rank_fusion = rank_based_fusion([p_cnn, p_mobile])

# إضافة النتائج
results.append(compute_metrics("Soft Fusion (Mean)", soft_mean, y_test))
results.append(compute_metrics("Hard Fusion (Max)", hard_max, y_test))
results.append(compute_metrics("Weighted Fusion (Optimized)", weighted, y_test))
results.append(compute_metrics("Median Fusion", median_f, y_test))
results.append(compute_metrics("Geometric Mean Fusion", geo_mean, y_test))
results.append(compute_metrics("Min Rule Fusion", min_rule, y_test))
results.append(compute_metrics("Product Rule Fusion", product_f, y_test))
results.append(compute_metrics("Rank-Based Fusion", rank_fusion, y_test))

# ====================== عرض النتائج ======================
print("="*175)
print("📊 FINAL ENSEMBLE RESULTS")
print("="*175)
print(f"{'Method':<38} {'Val Acc':>8} {'Val Loss':>10} {'Prec-W':>9} {'Rec-W':>9} {'F1-W':>9} {'MCC':>9} {'ROC-AUC':>9}")
print("-"*175)

for r in results:
    print(f"{r['Method']:<38} {r['Val Acc']:>8.4f} {r['Val Loss']:>10.4f} "
          f"{r['Prec-W']:>9.4f} {r['Rec-W']:>9.4f} {r['F1-W']:>9.4f} "
          f"{r['MCC']:>9.4f} {r['ROC-AUC']:>9.4f}")

print("="*175)

✅ Shapes -> CNN: (4820, 32) | Mobile: (4820, 32) | y_test: (4820,)
✅ After slicing -> CNN: (4820, 32) | Mobile: (4820, 32)

🎯 Optimal Weights → CNN: 0.8213 | MobileNetV2: 0.0163

📊 FINAL ENSEMBLE RESULTS
Method                                  Val Acc   Val Loss    Prec-W     Rec-W      F1-W       MCC   ROC-AUC
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
CNN                                      0.9938     0.0348    0.9938    0.9938    0.9938    0.9936    0.9999
MobileNetV2                              0.9801     0.0722    0.9803    0.9801    0.9801    0.9794    0.9999
Soft Fusion (Mean)                       0.9934     0.0457    0.9934    0.9934    0.9934    0.9931    1.0000
Hard Fusion (Max)                        0.9929     0.0637    0.9930    0.9929    0.9930    0.9927    1.0000
Weighted Fusion (Optimized)              0.9938     0.0336    0.9938    0.9